# Step 8 — CM-under-noise matrix + claim checklist

Aggregates `runs/step7_cm_under_noise/*/metrics.json` from notebook 07.

**Claim (dev first; lock eval later):**  
ECAPA + AASIST (α=0.30) keeps much lower SASV-EER than ECAPA-only under additive noise, mainly by cutting SPF-EER.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "noise_gated_lib.py").exists():
    ROOT = ROOT / "replay-cnn-baseline" / "experiments" / "sasv_noise_gated"
sys.path.insert(0, str(ROOT))

from noise_gated_lib import RUNS_DIR, DEFAULT_SNRS_DB, snr_tag, save_json

STEP7 = RUNS_DIR / "step7_cm_under_noise"
OUT = RUNS_DIR / "step8_cm_noise_claim"
OUT.mkdir(parents=True, exist_ok=True)

rows = []
for snr in DEFAULT_SNRS_DB:
    tag = snr_tag(snr)
    for split in ("dev", "eval"):
        path = STEP7 / f"{split}_{tag}" / "metrics.json"
        if not path.exists():
            continue
        payload = json.loads(path.read_text(encoding="utf-8"))
        for system, metrics in payload.items():
            rows.append(
                {
                    "split": split,
                    "snr": tag,
                    "system": system,
                    "sasv_eer_percent": metrics.get("sasv_eer_percent"),
                    "sv_eer_percent": metrics.get("sv_eer_percent"),
                    "spf_eer_percent": metrics.get("spf_eer_percent"),
                }
            )

df = pd.DataFrame(rows)
df


### SASV-EER (%) matrix


In [ ]:
if df.empty:
    print("No metrics — run notebook 07 first.")
else:
    order = [snr_tag(s) for s in DEFAULT_SNRS_DB]
    for split in sorted(df.split.unique()):
        pivot = (
            df[df.split == split]
            .pivot_table(index="snr", columns="system", values="sasv_eer_percent")
            .reindex(order)
        )
        print(f"=== {split} SASV-EER % ===")
        display(pivot.round(3))
        pivot.to_csv(OUT / f"matrix_sasv_eer_{split}.csv")

    spf = (
        df[df.split == "dev"]
        .pivot_table(index="snr", columns="system", values="spf_eer_percent")
        .reindex(order)
    )
    print("=== dev SPF-EER % ===")
    display(spf.round(3))
    spf.to_csv(OUT / "matrix_spf_eer_dev.csv")


### Plot


In [ ]:
if not df.empty:
    order = [snr_tag(s) for s in DEFAULT_SNRS_DB]
    sub = df[df.split == "dev"]
    fig, ax = plt.subplots(figsize=(7, 4))
    for system in sorted(sub.system.unique()):
        part = sub[sub.system == system].set_index("snr").reindex(order)
        ax.plot(order, part["sasv_eer_percent"], marker="o", label=system)
    ax.set_xlabel("Test condition")
    ax.set_ylabel("SASV-EER (%)")
    ax.set_title("Dev — ECAPA ± CM under noise")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig_path = OUT / "sasv_eer_vs_snr_dev.png"
    fig.savefig(fig_path, dpi=150)
    print("wrote", fig_path)
    plt.show()


### Claim checklist (dev smoke/full)

Set flags from **your** numbers. Locked eval comes later.


In [ ]:
def _eer(split, snr, system):
    hit = df[(df.split == split) & (df.snr == snr) & (df.system == system)]
    if hit.empty:
        return None
    return float(hit.iloc[0]["sasv_eer_percent"])


def _spf(split, snr, system):
    hit = df[(df.split == split) & (df.snr == snr) & (df.system == system)]
    if hit.empty:
        return None
    return float(hit.iloc[0]["spf_eer_percent"])


noise_tags = [snr_tag(s) for s in DEFAULT_SNRS_DB if s is not None]
b1_beats_b0 = True
spf_drops = True
details = {}
for tag in noise_tags:
    b0 = _eer("dev", tag, "B0")
    b1 = _eer("dev", tag, "B1_w030")
    if b0 is None or b1 is None:
        b1_beats_b0 = False
        spf_drops = False
        details[tag] = {"missing": True}
        continue
    details[tag] = {
        "B0": b0,
        "B1_w030": b1,
        "delta": b1 - b0,
        "spf_B0": _spf("dev", tag, "B0"),
        "spf_B1_w030": _spf("dev", tag, "B1_w030"),
    }
    if not (b1 < b0):
        b1_beats_b0 = False
    sp0 = details[tag]["spf_B0"]
    sp1 = details[tag]["spf_B1_w030"]
    if sp0 is None or sp1 is None or not (sp1 < sp0):
        spf_drops = False

checklist = {
    "B1_w030_beats_B0_under_noise_dev": bool(b1_beats_b0),
    "B1_w030_cuts_SPF_under_noise_dev": bool(spf_drops),
    "per_snr": details,
    "eval_locked": False,
    "decision": (
        "PASS (dev) — CM helps under noise; consider locked eval next"
        if b1_beats_b0 and spf_drops
        else "PENDING/FAIL — inspect per_snr deltas; do not force the claim"
    ),
}
save_json(OUT / "claim_checklist.json", checklist)
checklist


### Done

- Matrices / plot under `runs/step8_cm_noise_claim/`
- If checklist PASS on full **dev**, set `SMOKE=False` in 07 (if not already), then run **eval** once and freeze
